# Explicabilidad de Modelos de IA (XAI) en Ciberseguridad

## Importancia de la Explicabilidad

En ciberseguridad, entender _por qué_ un modelo tomó una decisión es crítico:
- **Auditoría y cumplimiento**: GDPR, NIS2 requieren trazabilidad
- **Validación de analistas**: Confianza en modelos de detección
- **Detección de sesgo**: Identificar factores discriminatorios
- **Forense digital**: Generar evidencia legal

Este cuaderno usa **SHAP** para explicar predicciones de malware y anomalías.


---

## SHAP: Explicación de Predicciones mediante Teoría de Juegos


In [ ]:
# Listing 7.1: Explicación de predicciones con SHAP

import shap
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# 1. Entrenar modelo
df = pd.read_csv('data/file_features.csv').dropna()
X, y = df.drop('label', axis=1), df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

print(f"[OK] Modelo Random Forest entrenado.")
print(f"     Accuracy: {rf.score(X_test, y_test):.4f}")
print(f"     Características: {list(X.columns)}")

In [ ]:
# 2. Calcular valores SHAP
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

print(f"[OK] Valores SHAP calculados para {len(X_test)} muestras.")

In [ ]:
# 3. Gráfico de resumen (importancia global)
shap.summary_plot(
    shap_values[1], X_test,
    plot_type='bar',
    show=False
)
plt.title('Importancia SHAP - Detección de Malware')
plt.tight_layout()
plt.savefig('data/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Gráfico de puntos SHAP (distribución de impacto por característica)
shap.summary_plot(
    shap_values[1], X_test,
    show=False
)
plt.title('SHAP - Distribución de impacto por característica')
plt.tight_layout()
plt.savefig('data/shap_dot_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4. Explicar una muestra individual
muestra_idx = 0
shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][muestra_idx, :],
    X_test.iloc[muestra_idx, :],
    matplotlib=True,
    show=False
)
plt.title('Explicación individual - Muestra 0')
plt.tight_layout()
plt.savefig('data/shap_force_plot.png', dpi=150, bbox_inches='tight')
plt.show()

# Mostrar detalle de la muestra explicada
pred = rf.predict(X_test.iloc[[muestra_idx]])[0]
proba = rf.predict_proba(X_test.iloc[[muestra_idx]])[0]
print(f"\nMuestra {muestra_idx}:")
print(f"  Predicción: {'Malicious' if pred == 1 else 'Benign'}")
print(f"  Probabilidad: Benign={proba[0]:.3f}, Malicious={proba[1]:.3f}")
print(f"  Valores de características:")
for col, val in X_test.iloc[muestra_idx].items():
    print(f"    {col:<25}: {val}")

### Gráficos de Dependencia: Relación Característica-Predicción


In [ ]:
# Gráfico de dependencia para las 2 características más importantes
import numpy as np

mean_shap = np.abs(shap_values[1]).mean(axis=0)
top_features = X_test.columns[np.argsort(mean_shap)[::-1][:2]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, feat in enumerate(top_features):
    plt.sca(axes[i])
    shap.dependence_plot(
        feat, shap_values[1], X_test,
        ax=axes[i], show=False
    )
    axes[i].set_title(f'Dependencia SHAP: {feat}')

plt.tight_layout()
plt.savefig('data/shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

---

## Documentación de Artefactos Generados


In [ ]:
print("=== Resumen de artefactos generados (XAI) ===")
print("Gráficos SHAP:")
print("  data/shap_summary.png       - Importancia global (barras)")
print("  data/shap_dot_summary.png   - Distribución de impacto (puntos)")
print("  data/shap_force_plot.png    - Explicación individual")
print("  data/shap_dependence.png    - Gráficos de dependencia")
print("\n[OK] Capítulo 7 completado. Los analistas pueden usar estas")
print("     visualizaciones para auditar las decisiones de los modelos.")